# ⚡ **E-Scoot Charging Network Optimization**
**Author:** Vinit Vijaykumar Adke
**Domain:** Supply Chain Network Design / Operations Research

---

### 📌 **Executive Summary**
**Business Problem:** The rapid adoption of E-Scooters at the ASU Tempe campus has outpaced charging infrastructure, leading to low availability and service gaps.
**Objective:** Determine the optimal **Facility Location** and **Capacity Allocation** for new charging stations to meet rider demand while minimizing installation costs.

### 📊 **Methodology**
* **Technique:** Mixed-Integer Linear Programming (MILP).
* **Solver:** Gurobi Optimizer (Python API).
* **Constraints Modeled:**
    1.  **Budget Cap:** Total project cost must not exceed $35,000.
    2.  **Service Level:** 100% of demand in all 6 campus zones must be met.
    3.  **Equity:** At least 70% coverage for residential housing zones.
    4.  **Physical Capacity:** Max 5–9 chargers per specific location.

### 🚀 **Key Results**
* **Optimal Design:** Selected **4 strategic locations** (Hassayampa, Sun Devil Fitness, Palo Verde, Engineering Center) out of 8 candidates.
* **Cost Efficiency:** Achieved full coverage for **$28,350**, coming in **19% under** the $35,000 budget.
* **Impact:** The model provides a scalable framework for future campus mobility infrastructure planning.

---

## Step 1: Input Data
We define the problem inputs including potential locations, demand zones, costs, capacity, demand, and the coverage matrix.

In [1]:
# Import necessary library
from gurobipy import Model, GRB, quicksum

# Define candidate locations and zones
locations = ['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8']
zones = ['Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6']

# Define costs
fixed_cost = {'L1': 3200, 'L2': 3500, 'L3': 2800, 'L4': 3000,
              'L5': 2900, 'L6': 3600, 'L7': 3400, 'L8': 3100}
variable_cost = {'L1': 600, 'L2': 650, 'L3': 700, 'L4': 750,
                 'L5': 720, 'L6': 580, 'L7': 630, 'L8': 670}
max_chargers = {'L1': 6, 'L2': 9, 'L3': 7, 'L4': 5,
                'L5': 8, 'L6': 7, 'L7': 8, 'L8': 5}

# Demand in each zone
demand = {'Z1': 9, 'Z2': 6, 'Z3': 13, 'Z4': 11, 'Z5': 14, 'Z6': 7}
total_demand = sum(demand.values())

# Coverage matrix
coverage = {
    ('L1','Z1'):0, ('L1','Z2'):1, ('L1','Z3'):1, ('L1','Z4'):0, ('L1','Z5'):1, ('L1','Z6'):1,
    ('L2','Z1'):1, ('L2','Z2'):1, ('L2','Z3'):0, ('L2','Z4'):0, ('L2','Z5'):1, ('L2','Z6'):0,
    ('L3','Z1'):0, ('L3','Z2'):0, ('L3','Z3'):1, ('L3','Z4'):1, ('L3','Z5'):1, ('L3','Z6'):0,
    ('L4','Z1'):1, ('L4','Z2'):0, ('L4','Z3'):1, ('L4','Z4'):0, ('L4','Z5'):1, ('L4','Z6'):0,
    ('L5','Z1'):1, ('L5','Z2'):1, ('L5','Z3'):0, ('L5','Z4'):0, ('L5','Z5'):1, ('L5','Z6'):0,
    ('L6','Z1'):0, ('L6','Z2'):1, ('L6','Z3'):1, ('L6','Z4'):1, ('L6','Z5'):0, ('L6','Z6'):1,
    ('L7','Z1'):1, ('L7','Z2'):0, ('L7','Z3'):1, ('L7','Z4'):1, ('L7','Z5'):1, ('L7','Z6'):1,
    ('L8','Z1'):0, ('L8','Z2'):0, ('L8','Z3'):1, ('L8','Z4'):0, ('L8','Z5'):1, ('L8','Z6'):1,
}

# Budget limit
budget = 35000


## Step 2: Define Decision Variables
We create integer and binary variables for the number of chargers and installation status respectively.

In [4]:
# Create the model
model = Model("Ebike_Charging_Station_Optimization")

# Integer variable for number of chargers
x = model.addVars(locations, vtype=GRB.INTEGER, name="ChargersInstalled")

# Binary variable for whether a station is installed
y = model.addVars(locations, vtype=GRB.BINARY, name="StationOpen")


Restricted license - for non-production use only - expires 2026-11-23


## Step 3: Define Objective Function
We minimize the total installation cost, which includes both fixed and variable components.

In [7]:
# Objective: Minimize total installation cost
model.setObjective(quicksum(fixed_cost[i]*y[i] + variable_cost[i]*x[i] for i in locations), GRB.MINIMIZE)


## Step 4: Add Constraints
We add five sets of constraints: installation logic, zone demand, budget, dorm fairness, and load balancing.

In [10]:
# Constraint 1: Chargers can only be installed if the station is installed
for i in locations:
    model.addConstr(x[i] <= max_chargers[i] * y[i], name=f"MaxCap_{i}")

# Constraint 2: Each zone must meet or exceed its demand
for j in zones:
    model.addConstr(quicksum(coverage[i, j] * x[i] for i in locations) >= demand[j], name=f"Demand_{j}")

# Constraint 3: Total installation cost must not exceed budget
model.addConstr(quicksum(fixed_cost[i] * y[i] + variable_cost[i] * x[i] for i in locations) <= budget, name="Budget")

# Constraint 4: At least 70% of dorm area demand must be covered by L2 and L5
model.addConstr(x['L2'] + x['L5'] >= 0.7 * (demand['Z1'] + demand['Z2']), name="ResidentialCoverage")

# Constraint 5: No single location serves more than 30 students
for i in locations:
    model.addConstr(quicksum(coverage[i, j] * x[i] for j in zones) <= 30, name=f"LoadBalance_{i}")


## Step 5: Solve and Display Results
We solve the model and then print the decision for each location in a readable format.

In [13]:
# Optimize the model
model.optimize()

# Location names
asu_locations = {
    'L1': "Hayden Library", 'L2': "Hassayampa Dormitory", 'L3': "Sun Devil Fitness Center",
    'L4': "MU Food Court", 'L5': "Palo Verde Dormitory", 'L6': "Engineering Center",
    'L7': "Health Services", 'L8': "W. P. Carey Business School"
}

# Print output
print("\n📊 Charging Station Deployment Plan (ASU Tempe Campus):")
print("-" * 65)
print(f"{'Location':<35} {'Installed':<12} {'Chargers':<10}")
print("-" * 65)
for i in locations:
    print(f"{asu_locations[i]:<35} {int(y[i].X):<12} {int(x[i].X):<10}")
print("-" * 65)
print(f"\n✅ Total Installation Cost = ${model.ObjVal:,.2f}")


Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.3.0 24D70)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 24 rows, 16 columns and 70 nonzeros
Model fingerprint: 0xc7eb7674
Variable types: 0 continuous, 16 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+03]
  Objective range  [6e+02, 4e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e+00, 4e+04]
Found heuristic solution: objective 31990.000000
Presolve removed 14 rows and 4 columns
Presolve time: 0.00s
Presolved: 10 rows, 12 columns, 37 nonzeros
Found heuristic solution: objective 31570.000000
Variable types: 0 continuous, 12 integer (6 binary)

Root relaxation: objective 2.795000e+04, 8 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 27950.0000  